# Project Setup

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [6]:
from sklearn.metrics import accuracy_score, recall_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset

In [7]:
#Store the file path inside a variable named DATA_PATH
DATA_PATH = "/Users/will/Desktop/Year 3 - Semester 1/Artificial Intelligence and Machine Learning/CMP6202-coursework/data/raw/student_dropout.csv"

In [8]:
#Use pandas 'read_csv' function to read the data, store in variable named 'df'
#Important so original data isn't being modified, can reload this cell to restore dataset to original state
df = pd.read_csv(DATA_PATH)

In [9]:
df.head(10)

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate
5,2,12,1,17,0,12,1,22,27,10,...,0,5,17,5,11.500000,5,16.2,0.3,-0.92,Graduate
6,1,1,1,12,1,1,1,13,28,8,...,0,8,8,8,14.345000,0,15.5,2.8,-4.06,Graduate
7,1,9,4,11,1,1,1,22,27,10,...,0,5,5,0,0.000000,0,15.5,2.8,-4.06,Dropout
8,1,1,3,10,1,1,15,1,1,10,...,0,6,7,6,14.142857,0,16.2,0.3,-0.92,Graduate
9,1,1,1,10,1,1,1,1,14,5,...,0,6,14,2,13.500000,0,8.9,1.4,3.51,Dropout


In [10]:
#Dataset has 4424 rows, 35 columns
print(df.shape)

(4424, 35)


# Checking dataset for any missing values

In [11]:
#Check for all missing values accross all columns, store in variable 'missing_values'
#Print the sum of all the missing values
missing_values = df.isnull().sum()
print(missing_values.sum())

0


In [12]:
#Check for any missing values (returns True or False)
print(df.isnull().values.any())

False


# Dropping any rows with missing values

 My dataset doesn't contain any rows with missing values, but below is the code that I would use if I were to drop rows. I also understand that dropping rows with missing values isn't always the best approach, methods such as IMPUTATION should be considered

In [13]:
df = df.dropna()

In [14]:
#Same amount of rows and columns as previously (before dropping missing values)
print(df.shape)

(4424, 35)


# Checking for columns that store text

In [15]:
#Checking all of the columns within my dataset to see what 'types' they are
print(df.dtypes)

Marital status                                      int64
Application mode                                    int64
Application order                                   int64
Course                                              int64
Daytime/evening attendance                          int64
Previous qualification                              int64
Nacionality                                         int64
Mother's qualification                              int64
Father's qualification                              int64
Mother's occupation                                 int64
Father's occupation                                 int64
Displaced                                           int64
Educational special needs                           int64
Debtor                                              int64
Tuition fees up to date                             int64
Gender                                              int64
Scholarship holder                                  int64
Age at enrollm

In [16]:
#Beacause my dataset is 'multiclass classification', I want to keep the targer value as a String (enrolled, dropout, grduate)
print(df.select_dtypes(include=['object', 'string']).columns)

Index(['Target'], dtype='str')


If I wanted to convert any columns or my target value into a numerical value I would use 'LabelEncoder'. Please see the commented code below. 

In [18]:
# Converting specific columns into numerical values

# encoder = LabelEncoder()
# df['columnname'] = encoder.fit_transform(encoder['columnname'])

In [19]:
# Converting targer value into numerical values
# df['Target'] = encoder.fit_transform(encoder['Target])

# Seperating the dataset between X and Y (X = prediction columns, Y = target column)

In [21]:
#X = dataset with target column dropped/removed
X = df.drop(columns=['Target'])

#Y = target column
y = df['Target']


In [23]:
print(X.head(2))

   Marital status  Application mode  Application order  Course  \
0               1                 8                  5       2   
1               1                 6                  1      11   

   Daytime/evening attendance  Previous qualification  Nacionality  \
0                           1                       1            1   
1                           1                       1            1   

   Mother's qualification  Father's qualification  Mother's occupation  ...  \
0                      13                      10                    6  ...   
1                       1                       3                    4  ...   

   Curricular units 1st sem (without evaluations)  \
0                                               0   
1                                               0   

   Curricular units 2nd sem (credited)  Curricular units 2nd sem (enrolled)  \
0                                    0                                    0   
1                                 

In [35]:
print(y.head(2))

0     Dropout
1    Graduate
Name: Target, dtype: str


# Create the training and test splits

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training students: ", len(X_train))
print("Test students: ", len(X_test))

Training students:  3539
Test students:  885


# Baseline Accuracy

In [38]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_score = DummyClassifier(strategy="most_frequent")
base_scores = cross_val_score(baseline_score, X_train, y_train, cv=skf)

print("Baseline Accuracy: ", round(base_scores.mean(), 3))

Baseline Accuracy:  0.499


# Cross Validation Baseline


In [39]:
model = GaussianNB()

scores = cross_val_score(model, X_train, y_train, cv=skf)

print("5 Scores: ", scores.round(3))
print("Average of 5 scores: ", round(scores.mean(), 3))
print("Difference in scores: ", round(scores.std(), 3))


5 Scores:  [0.689 0.703 0.695 0.658 0.678]
Average of 5 scores:  0.685
Difference in scores:  0.016


# Decision Tree

In [36]:
dt = DecisionTreeClassifier()

dt_scores = cross_val_score(dt, X_train, y_train, cv=skf, scoring="accuracy")
dt_avg = dt_scores.mean().round(2)

print(dt_scores)
print(dt_avg)

[0.6779661  0.67937853 0.67655367 0.69915254 0.67326733]
0.68


# Random Forest

In [35]:
rf = RandomForestClassifier(random_state=42)

rf_scores = cross_val_score(rf, X_train, y_train, cv=skf, scoring="accuracy")
rf_avg = rf_scores.mean().round(2)

print(rf_scores)
print(rf_avg)

[0.78531073 0.75564972 0.78954802 0.75847458 0.77227723]
0.77


# Model Scores

In [45]:
models = pd.DataFrame({
    "models": ['Baseline', 'Decision Tree', 'Random Forest'],
    "accuracy": [scores, dt_avg, rf_avg]
})

models.sort_values(by='accuracy', ascending=False)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()